# Week 5 — Agents, tools, MCP, state, and approval

Compare in-process, prompt, Hosted, and self-hosted agents. Hosting and protocol are separate choices. Give every tool a narrow schema and deterministic authorization; the model's request is never authorization.

In [ ]:
import importlib.util
import sys
from pathlib import Path

curriculum_root = next(
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / "foundry-curriculum")
    if (candidate / "notebook_setup.py").is_file()
)
spec = importlib.util.spec_from_file_location(
    "foundry_curriculum_setup", curriculum_root / "notebook_setup.py"
)
helpers = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = helpers
spec.loader.exec_module(helpers)
session = helpers.load_session(curriculum_root)
session.safe_summary()

## Authorization is deterministic; the model's request is not

A tool call arrives as text the model generated, and text cannot authorize
anything. Every call therefore passes through a table the model does not
control, which answers two separate questions:

1. **May this caller use this tool at all?** Answered by group membership.
2. **Does this tool change the world?** Answered by `side_effect`, and a
   side-effecting call needs a second, human answer before it runs.

Collapsing those two questions into one is the mistake that turns a helpful
agent into an incident. Read the next cell looking for where they stay apart.

In [ ]:
TOOL_POLICY = {
    "search_curriculum": {
        "side_effect": False,
        "allowed_groups": {"foundry-learners"},
        "timeout_seconds": 5,
    },
    "publish_release": {
        "side_effect": True,
        "allowed_groups": {"release-owners"},
        "timeout_seconds": 10,
    },
}


def authorize_tool_call(name, caller_groups, approved=False):
    policy = TOOL_POLICY[name]
    if not (policy["allowed_groups"] & set(caller_groups)):
        return False, "caller is not authorized"
    if policy["side_effect"] and not approved:
        return False, "human approval is required"
    return True, "authorized"


assert authorize_tool_call("search_curriculum", {"foundry-learners"})[0]
assert not authorize_tool_call("publish_release", {"release-owners"})[0]
assert authorize_tool_call(
    "publish_release", {"release-owners"}, approved=True
)[0]
TOOL_POLICY

### What you just saw

Three assertions, three different outcomes. The read succeeds outright. The
release publish is **denied even for a member of `release-owners`** — that is
the line carrying the lesson. Group membership answered "may this caller",
and `side_effect: True` still demanded a separate answer to "should this
happen now". The third assertion passes only because `approved=True` supplies
that answer from outside the model.

### Change this and re-run

Drop `approved=True` from the third assertion. It fails, and that failure *is*
the control: an agent asking to publish a release is making a request, not
granting itself permission.

Now set `"side_effect": False` on `publish_release` and re-run. It passes with
no human in the loop. That one flag is the entire difference between a tool
that can be automated and one that cannot, which is why it belongs in a policy
table under change control rather than in a prompt.

## Agent threat model

Document goal hijacking, tool misuse, identity abuse, MCP supply-chain and data-egress risk, memory poisoning, cascading failure, unbounded consumption, and human over-trust. Add step/token/time budgets, endpoint allow-lists, idempotency keys, a memory retention policy, and an interrupt before side effects.

## Hosting and evaluation are separate choices

This notebook opened by separating hosting from protocol. Release gating adds
a third independent axis: **the framework you build the agent with does not
decide how the agent is evaluated.**

The AgentOps Accelerator makes this concrete. It ships no adapter for
LangGraph, Semantic Kernel, or Microsoft Agent Framework — deliberately.
Any agent reachable over HTTP is gated through one contract:

```text
POST /chat            {"message": "<row input>"}
200 application/json  {"text": "<answer>", "tool_calls": [...], "context": [...]}
```

A LangGraph graph, an Agent Framework workflow, and a hand-written FastAPI
route are indistinguishable to the gate as long as they serve that shape. The
absence of a framework adapter is the design, not a gap.

The next cell models a durable graph **as data**, so the lesson survives
without adding a framework dependency to this curriculum. The executable
reference implementation lives in the repository at
`templates/agent-app/template/recipes/langgraph/graph.py`.

In [ ]:
def interrupts_before_side_effects(graph):
    """Return the side-effecting nodes reachable without a prior interrupt."""

    unguarded = []
    for edge_from, edge_to in graph["edges"]:
        target = graph["nodes"][edge_to]
        if target["side_effect"] and not graph["nodes"][edge_from]["interrupt"]:
            unguarded.append(f"{edge_from} -> {edge_to}")
    return unguarded


support_graph = {
    "nodes": {
        "propose": {"side_effect": False, "interrupt": False},
        "await_approval": {"side_effect": False, "interrupt": True},
        "execute": {"side_effect": True, "interrupt": False},
    },
    "edges": [
        ("propose", "await_approval"),
        ("await_approval", "execute"),
    ],
    "idempotency_key": "request_id",
    "checkpointed_state": ["request", "proposed_action", "approved", "result"],
}

assert interrupts_before_side_effects(support_graph) == []
# The retry key must survive the checkpoint, or "resume" means "do it twice".
assert support_graph["idempotency_key"] == "request_id"
assert "request" in support_graph["checkpointed_state"]

# The same shape served over HTTP is what a release gate actually talks to.
gate_target = {
    "agent": "https://<your-agent-host>/chat",
    "protocol": "http-json",
    "request_field": "message",
    "response_field": "text",
    "tool_calls_field": "tool_calls",
    "auth_header_env": "APP_API_TOKEN",
}

assert not gate_target["auth_header_env"].startswith(("http", "sk-", "Bearer"))
{"unguarded_side_effects": interrupts_before_side_effects(support_graph)}

### What you just saw

`interrupts_before_side_effects` returned an empty list: no edge reaches
`execute` without passing through `await_approval` first. That is the same
control as `TOOL_POLICY`, expressed in graph form rather than table form —
which is the point. The control is architectural, so it does not depend on
which framework compiles the graph.

Two supporting details worth naming:

- **`idempotency_key` is part of checkpointed state.** A durable graph that
  resumes after approval will re-enter `execute`. Without a key carried in the
  checkpoint, "resume" and "do it twice" are the same code path.
- **`auth_header_env` names an environment variable, never a token.** The
  assertion enforces that. The gate reads the variable at run time; a literal
  in configuration would be a secret in Git.

### Change this and re-run

Add `("propose", "execute")` to `edges` — a plausible "fast path" someone adds
under deadline pressure. `interrupts_before_side_effects` now reports it, and
the assertion fails. The graph still runs correctly in the happy case; only
the check notices that the approval step became optional.

## The gate configuration this produces

`gate_target` above is the notebook form of an `agentops.yaml` HTTP target.
The file form, for reference:

```yaml
version: 1
agent: https://<your-agent-host>/chat
protocol: http-json          # inferred from the URL shape; shown for clarity
request_field: message       # default "message"
response_field: text         # default "text"; dot-paths are supported
tool_calls_field: tool_calls # what makes trajectory evaluation possible
response_fields:
  context: $response.context # captures retrieved context for groundedness
response_mode: json          # json | sse | text
auth_header_env: APP_API_TOKEN
dataset: .agentops/data/curriculum_cases.jsonl
```

Three failure modes worth recognising before you meet them:

- **A JSON parse error from a working endpoint** usually means the endpoint
  streams. Set `response_mode: sse` (or `text`).
- **An empty `auth_header_env`** is a hard error, not a silent anonymous call.
- **A missing `tool_calls_field`** silently reduces the gate to final-answer
  scoring. The agent can reach the right answer through a forbidden tool and
  still pass, which is exactly the failure `TOOL_POLICY` exists to prevent.

Notebook `08_agentops_release_gate.ipynb` turns this configuration into an
enforced CI gate.

## Exit criteria

Demonstrate an allowed read, a denied unauthorized call, a blocked side effect
awaiting approval, and an idempotent retry. Then show the same agent behind an
HTTP contract, and name which field of that contract makes each of those four
behaviours observable to a release gate.

Treat multi-agent orchestration as an advanced core lab; keep A2A-specific
work optional when its required components are preview.